In [4]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [5]:
class MovieRecommender:
    def __init__(self, movies_df, ratings_df):
        self.movies = movies_df
        self.ratings = ratings_df
        self.user_ids = self.ratings['userId'].unique().tolist()
        self._build_models()

    def _build_models(self):
        # Content-based model
        tfidf = TfidfVectorizer(stop_words='english')
        self.movies['content'] = (
            self.movies['title'].fillna('') + ' ' +
            self.movies['genres'].fillna('')
        )
        self.tfidf_matrix = tfidf.fit_transform(self.movies['content'])
        self.content_similarity = cosine_similarity(self.tfidf_matrix)

        # Collaborative model
        self.user_movie_matrix = self.ratings.pivot_table(
            index='userId',
            columns='movieId',
            values='rating'
        ).fillna(0)

        kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
        user_clusters = kmeans.fit_predict(self.user_movie_matrix)
        self.user_movie_matrix['cluster'] = user_clusters

        self.knn = NearestNeighbors(metric='cosine', algorithm='brute')
        self.knn.fit(self.user_movie_matrix.drop('cluster', axis=1))

    def _get_user_genre_preferences(self, user_id, top_n=5):
        """Get user's top genre preferences based on their ratings"""
        if user_id not in self.ratings['userId'].values:
            return {}

        user_ratings = self.ratings[self.ratings['userId'] == user_id]
        user_movies = user_ratings.merge(self.movies[['movieId', 'genres']], on='movieId')

        # Calculate genre preferences
        genre_ratings = {}
        for _, row in user_movies.iterrows():
            if pd.notna(row['genres']):
                for genre in row['genres'].split('|'):
                    if genre not in genre_ratings:
                        genre_ratings[genre] = []
                    genre_ratings[genre].append(row['rating'])

        # Average ratings per genre
        genre_avg = {genre: np.mean(ratings) for genre, ratings in genre_ratings.items()}

        # Sort by average rating
        sorted_genres = sorted(genre_avg.items(), key=lambda x: x[1], reverse=True)
        return dict(sorted_genres[:top_n])

    def content_recommend(self, movie_title, n=10):
        if movie_title not in self.movies['title'].values:
            popular = self.ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
            return self.movies[self.movies['movieId'].isin(popular.index)][['movieId', 'title', 'genres']]

        idx = self.movies[self.movies['title'] == movie_title].index[0]
        sim_scores = list(enumerate(self.content_similarity[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        sim_scores = sim_scores[1:n+1]
        movie_indices = [i[0] for i in sim_scores]

        return self.movies.iloc[movie_indices][['title', 'genres', 'movieId']]

    def collaborative_recommend(self, user_id, n=10):
        if user_id not in self.user_movie_matrix.index:
            popular = self.ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
            return self.movies[self.movies['movieId'].isin(popular.index)][['title', 'genres', 'movieId']]

        user_vector = self.user_movie_matrix.drop('cluster', axis=1).loc[user_id]

        if user_vector.sum() == 0:
            popular = self.ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
            return self.movies[self.movies['movieId'].isin(popular.index)][['title', 'genres', 'movieId']]

        distances, indices = self.knn.kneighbors([user_vector], n_neighbors=min(6, len(self.user_movie_matrix)))
        neighbors = self.user_movie_matrix.index[indices.flatten()]

        neighbor_ratings = self.ratings[self.ratings['userId'].isin(neighbors)]

        # Movies already watched by target user
        watched_movies = set(
            self.ratings[self.ratings['userId'] == user_id]['movieId']
        )

        top_movies = (
            neighbor_ratings.groupby('movieId')['rating']
            .mean()
            .sort_values(ascending=False)
        )

        top_movies = top_movies[~top_movies.index.isin(watched_movies)].head(n)

        return self.movies[self.movies['movieId'].isin(top_movies.index)][['movieId', 'title', 'genres']]

    def hybrid_recommend(self, user_id, movie_title, n=10):
        content_recs = self.content_recommend(movie_title, n*2)
        collab_recs = self.collaborative_recommend(user_id, n*2)

        if content_recs.empty and collab_recs.empty:
              popular = self.ratings.groupby('movieId')['rating'].mean().sort_values(ascending=False).head(n)
              return self.movies[self.movies['movieId'].isin(popular.index)][['movieId', 'title', 'genres']]

        # Add recommendation type and score
        if not content_recs.empty:
            content_recs['recommendation_type'] = 'content-based'
            content_recs['score'] = 1.0 - np.arange(len(content_recs)) / len(content_recs)
        else:
            content_recs = pd.DataFrame(columns=['movieId', 'title', 'genres', 'recommendation_type', 'score'])

        if not collab_recs.empty:
            collab_recs['recommendation_type'] = 'collaborative'
            collab_recs['score'] = 1.0 - np.arange(len(collab_recs)) / len(collab_recs)
        else:
            collab_recs = pd.DataFrame(columns=['movieId', 'title', 'genres', 'recommendation_type', 'score'])

        hybrid = pd.concat([content_recs, collab_recs])

        hybrid = (
            hybrid.groupby(['movieId', 'title', 'genres'])['score']
            .sum()
            .reset_index()
            .sort_values('score', ascending=False)
        )

        return hybrid.head(n)

    def explain_recommendation(self, user_id, movie_title, n=5):
        """
        Get recommendations with explanations
        """
        recs = self.hybrid_recommend(user_id, movie_title, n)
        explanations = []
        confidence_scores = []

        user_genres = self._get_user_genre_preferences(user_id)

        for _, movie in recs.iterrows():
            movie_genres = set(movie['genres'].split('|')) if pd.notna(movie['genres']) else set()

            # Simplified approach for explanation, without granular type tracking post-aggregation:
            explanation = "Recommended based on a hybrid approach"
            confidence = movie['score'] / (1.0 * 2) # Max possible combined score is 2.0 (1.0 from each if n=1)

            if user_id in self.ratings['userId'].values:
                user_ratings = self.ratings[self.ratings['userId'] == user_id]
                watched_movies = user_ratings.merge(self.movies[['movieId', 'title']], on='movieId')
                # Try to find a content similarity explanation
                if movie_title and movie_title in self.movies['title'].values:
                    input_movie_idx = self.movies[self.movies['title'] == movie_title].index[0]
                    current_movie_idx = self.movies[self.movies['movieId'] == movie['movieId']].index
                    if not current_movie_idx.empty:
                        current_movie_idx = current_movie_idx[0]
                        content_sim = self.content_similarity[input_movie_idx, current_movie_idx]
                        if content_sim > 0.5: # Arbitrary threshold for "similar"
                            explanation = f"Similar to '{movie_title}' in content (similarity: {content_sim:.2f})"
                            confidence = max(confidence, content_sim)

                # Try to find collaborative explanation (users with similar tastes)
                similar_users_liked = self.ratings[
                    (self.ratings['movieId'] == movie['movieId']) &
                    (self.ratings['rating'] >= 4)
                ].merge(user_ratings, on='userId', how='inner')

                if not similar_users_liked.empty:
                    avg_sim_rating = similar_users_liked['rating_x'].mean()
                    explanation = f"Rated {avg_sim_rating:.1f}/5 by users with similar tastes"
                    confidence = max(confidence, avg_sim_rating / 5)

                # Try to find genre preference explanation
                common_genres = movie_genres.intersection(set(user_genres.keys()))
                if common_genres:
                    explanation = f"You enjoy {', '.join(list(common_genres)[:2])} movies"
                    confidence = max(confidence, 0.7)

            explanations.append(explanation)
            confidence_scores.append(confidence)

        recs = recs.copy()
        recs['explanation'] = explanations
        recs['confidence'] = confidence_scores

        return recs.sort_values('confidence', ascending=False)[['title', 'genres', 'explanation', 'confidence']]

    def get_user_preferences(self, user_id):
        """
        Get detailed user preference analysis
        """
        if user_id not in self.ratings['userId'].values:
            return {'error': 'User not found'}

        user_ratings = self.ratings[self.ratings['userId'] == user_id]

        genre_preferences = self._get_user_genre_preferences(user_id, top_n=5)

        stats = {
            'total_ratings': len(user_ratings),
            'avg_rating': user_ratings['rating'].mean(),
            'rating_std': user_ratings['rating'].std(),
            'favorite_genres': genre_preferences,
            'rating_distribution': user_ratings['rating'].value_counts().to_dict()
        }

        top_rated = user_ratings.nlargest(5, 'rating')
        stats['top_movies'] = top_rated.merge(
            self.movies[['movieId', 'title']],
            on='movieId'
        )[['title', 'rating']].to_dict('records')

        return stats

In [6]:
# Load your data
movies = pd.read_csv('/content/mlfile/movies.csv')
ratings = pd.read_csv('/content/mlfile/ratings.csv')


In [13]:
display(movies.head())

,movieId,title,genres,content
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story (1995) Adventure|Animation|Children|...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji (1995) Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale (1995) Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [14]:
display(ratings.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
# Initialize recommender
recommender = MovieRecommender(movies, ratings)

# Get content-based recommendations
content_recs = recommender.content_recommend('Toy Story (1995)', n=10)

# Get collaborative recommendations
collab_recs = recommender.collaborative_recommend(user_id=1, n=10)

# Get hybrid recommendations
hybrid_recs = recommender.hybrid_recommend(user_id=1, movie_title='Toy Story (1995)', n=10)

# Get recommendations with explanations
explanations = recommender.explain_recommendation(user_id=1, movie_title='explanations', n=5)

# Get user preferences
preferences = recommender.get_user_preferences(user_id=1)



In [15]:
print("Preferances:")
print(preferences)

Preferances:
{'total_ratings': 232, 'avg_rating': np.float64(4.366379310344827), 'rating_std': 0.8000480467733447, 'favorite_genres': {'Film-Noir': np.float64(5.0), 'Animation': np.float64(4.689655172413793), 'Musical': np.float64(4.681818181818182), 'Children': np.float64(4.5476190476190474), 'Drama': np.float64(4.529411764705882)}, 'rating_distribution': {5.0: 124, 4.0: 76, 3.0: 26, 2.0: 5, 1.0: 1}, 'top_movies': [{'title': 'Seven (a.k.a. Se7en) (1995)', 'rating': 5.0}, {'title': 'Usual Suspects, The (1995)', 'rating': 5.0}, {'title': 'Bottle Rocket (1996)', 'rating': 5.0}, {'title': 'Rob Roy (1995)', 'rating': 5.0}, {'title': 'Canadian Bacon (1995)', 'rating': 5.0}]}


In [8]:
print("Explanation:")
print(explanations)

Explanation:
                               title  \
0                   Key Largo (1948)   
4                       Shrek (2001)   
1   Slumber Party Massacre II (1987)   
2  Slumber Party Massacre III (1990)   
3     Sorority House Massacre (1986)   

                                              genres  \
0                     Crime|Drama|Film-Noir|Thriller   
4  Adventure|Animation|Children|Comedy|Fantasy|Ro...   
1                                             Horror   
2                                             Horror   
3                                             Horror   

                              explanation  confidence  
0       You enjoy Drama, Film-Noir movies        0.70  
4    You enjoy Children, Animation movies        0.70  
1  Recommended based on a hybrid approach        0.50  
2  Recommended based on a hybrid approach        0.45  
3  Recommended based on a hybrid approach        0.40  


In [110]:
print("Content-based Recommendations:")
print(content_recs)

Content-based Recommendations:
                                      title  \
2355                     Toy Story 2 (1999)   
7355                     Toy Story 3 (2010)   
3595                        Toy, The (1982)   
2539  We're Back! A Dinosaur's Story (1993)   
26                      Now and Then (1995)   
4089                    Toy Soldiers (1991)   
1617          NeverEnding Story, The (1984)   
6194                       Wild, The (2006)   
1                            Jumanji (1995)   
12                             Balto (1995)   

                                                genres  movieId  
2355       Adventure|Animation|Children|Comedy|Fantasy     3114  
7355  Adventure|Animation|Children|Comedy|Fantasy|IMAX    78499  
3595                                            Comedy     4929  
2539              Adventure|Animation|Children|Fantasy     3400  
26                                      Children|Drama       27  
4089                                      Action|Drama 

In [111]:
print("collabrative-based Recommendations:")
print(collab_recs)

collabrative-based Recommendations:
      movieId                                              title  \
2498     3334                                   Key Largo (1948)   
3194     4306                                       Shrek (2001)   
4427     6539  Pirates of the Caribbean: The Curse of the Bla...   
4498     6659                                     Tremors (1990)   
4591     6820                                Ginger Snaps (2000)   
4604     6857               Ninja Scroll (Jûbei ninpûchô) (1995)   
4908     7360                            Dawn of the Dead (2004)   
4927     7387                            Dawn of the Dead (1978)   
5260     8636                                Spider-Man 2 (2004)   
5335     8874                           Shaun of the Dead (2004)   

                                                 genres  
2498                     Crime|Drama|Film-Noir|Thriller  
3194  Adventure|Animation|Children|Comedy|Fantasy|Ro...  
4427                    Action|Adventure|

In [112]:
print("Hybrid-based Recommendations:")
print(hybrid_recs)

Hybrid-based Recommendations:
    movieId                                  title  \
10     3300                     Pitch Black (2000)   
9      3114                     Toy Story 2 (1999)   
37    78499                     Toy Story 3 (2010)   
11     3334                       Key Largo (1948)   
13     3701                    Alien Nation (1988)   
19     4929                        Toy, The (1982)   
14     4180             Reform School Girls (1986)   
12     3400  We're Back! A Dinosaur's Story (1993)   
15     4306                           Shrek (2001)   
2        27                    Now and Then (1995)   

                                               genres  score  
10                             Horror|Sci-Fi|Thriller   1.00  
9         Adventure|Animation|Children|Comedy|Fantasy   1.00  
37   Adventure|Animation|Children|Comedy|Fantasy|IMAX   0.95  
11                     Crime|Drama|Film-Noir|Thriller   0.95  
13                        Crime|Drama|Sci-Fi|Thriller   0.90

In [10]:
def evaluate_recommender(model, test_ratings, recommender_type='hybrid', k=10):

    precisions = []
    recalls = []
    f1_scores = []
    hit_rates = []
    ndcgs = []

    users = test_ratings['userId'].unique()

    for user_id in users:

        relevant_movies = set(
            test_ratings[
                (test_ratings['userId'] == user_id) &
                (test_ratings['rating'] >= 4)
            ]['movieId']
        )

        if len(relevant_movies) == 0:
            continue

        try:

            if recommender_type == 'content':

                seed_movie_id = list(relevant_movies)[0]

                movie_row = model.movies[
                    model.movies['movieId'] == seed_movie_id
                ]

                if movie_row.empty:
                    continue

                seed_title = movie_row.iloc[0]['title']

                recs = model.content_recommend(
                    seed_title,
                    n=k
                )

            elif recommender_type == 'collaborative':

                recs = model.collaborative_recommend(
                    user_id,
                    n=k
                )

            elif recommender_type == 'hybrid':

                seed_movie_id = list(relevant_movies)[0]

                movie_row = model.movies[
                    model.movies['movieId'] == seed_movie_id
                ]

                if movie_row.empty:
                    continue

                seed_title = movie_row.iloc[0]['title']

                recs = model.hybrid_recommend(
                    user_id,
                    seed_title,
                    n=k
                )

            else:
                continue

        except:
            continue

        recommended = list(recs['movieId'])

        hits = len(
            set(recommended).intersection(relevant_movies)
        )

        precision = hits / k
        recall = hits / len(relevant_movies)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0
        )

        hit_rate = 1 if hits > 0 else 0

        dcg = sum(
            1 / np.log2(i + 2)
            for i, movie in enumerate(recommended)
            if movie in relevant_movies
        )

        idcg = sum(
            1 / np.log2(i + 2)
            for i in range(min(len(relevant_movies), k))
        )

        ndcg = dcg / idcg if idcg > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        hit_rates.append(hit_rate)
        ndcgs.append(ndcg)

    return {
        'Precision@10': np.mean(precisions),
        'Recall@10': np.mean(recalls),
        'F1@10': np.mean(f1_scores),
        'HitRate': np.mean(hit_rates),
        'NDCG@10': np.mean(ndcgs)
    }

In [11]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

model = MovieRecommender(
    movies_df=movies,
    ratings_df=train_ratings
)
content_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='content'
)

collab_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='collaborative'
)

hybrid_results = evaluate_recommender(
    model,
    test_ratings,
    recommender_type='hybrid'
)

In [12]:
print("CONTENT-BASED RESULTS")
for metric, value in content_results.items():
    print(f"{metric}: {value:.4f}")

print("\nCOLLABORATIVE RESULTS")
for metric, value in collab_results.items():
    print(f"{metric}: {value:.4f}")

print("\nHYBRID RESULTS")
for metric, value in hybrid_results.items():
    print(f"{metric}: {value:.4f}")

CONTENT-BASED RESULTS
Precision@10: 0.0109
Recall@10: 0.0109
F1@10: 0.0089
HitRate: 0.0935
NDCG@10: 0.0152

COLLABORATIVE RESULTS
Precision@10: 0.0326
Recall@10: 0.0493
F1@10: 0.0332
HitRate: 0.2471
NDCG@10: 0.0451

HYBRID RESULTS
Precision@10: 0.0287
Recall@10: 0.0370
F1@10: 0.0260
HitRate: 0.2304
NDCG@10: 0.0397
